# Session 3: INSERT, UPDATE & DELETE (DML Operations)

This interactive notebook explores **Data Manipulation Language (DML)** in SQL, including targeted record insertion, in-place updates, data integrity corrections, and safe deletions with `WHERE` filtering.

---

## Tasks Covered
- **Task 1:** Create `Playlist` table (`id`, `song_name`, `artist`, `duration`) & insert a favorite song.
- **Task 2:** Insert 3+ new tracks recently listened to on Spotify.
- **Task 3:** Fix an artist name typo (e.g., `'Arjit Singh'` -> `'Arijit Singh'`) using `UPDATE` with `WHERE`.
- **Task 4:** Delete short tracks (`duration < 120` seconds) using `DELETE` with `WHERE`.
- **Task 5:** Conditionally append `' (Remix)'` to all songs by `'AP Dhillon'` with `duration > 180` seconds.

### Task 1: Create Table 'Playlist' & Insert Favorite Song

In [ ]:
import sqlite3
import os
import pandas as pd

os.makedirs("database", exist_ok=True)
db_path = os.path.join("database", "spotify_playlist.db")
conn = sqlite3.connect(db_path)
cursor = conn.cursor()

cursor.execute("DROP TABLE IF EXISTS Playlist;")
create_sql = """
CREATE TABLE Playlist (
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    song_name VARCHAR(100) NOT NULL,
    artist VARCHAR(100) NOT NULL,
    duration INT NOT NULL,
    created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
);
"""
cursor.execute(create_sql)

# Insert initial favorite track (introducing deliberate typo 'Arjit Singh' for Task 3)
cursor.execute(
    "INSERT INTO Playlist (song_name, artist, duration) VALUES (?, ?, ?);",
    ("Tum Hi Ho", "Arjit Singh", 262)
)
conn.commit()

df_t1 = pd.read_sql_query("SELECT * FROM Playlist;", conn)
display(df_t1)

### Task 2: Insert Recently Listened Tracks from Spotify
We populate the playlist with varied tracks, including short interludes and multiple tracks by 'AP Dhillon'.

In [ ]:
recent_tracks = [
    ("Brown Munde", "AP Dhillon", 265),
    ("Intro (Interlude)", "Various Artists", 95),
    ("Excuses", "AP Dhillon", 195),
    ("With You", "AP Dhillon", 150),
    ("Kesariya", "Arijit Singh", 268)
]

cursor.executemany(
    "INSERT INTO Playlist (song_name, artist, duration) VALUES (?, ?, ?);",
    recent_tracks
)
conn.commit()

df_t2 = pd.read_sql_query("SELECT * FROM Playlist;", conn)
display(df_t2)

### Task 3: Fix Typo in Artist Name Using UPDATE with WHERE
Fixing `'Arjit Singh'` to `'Arijit Singh'`.

In [ ]:
update_typo_sql = """
UPDATE Playlist
SET artist = 'Arijit Singh'
WHERE artist = 'Arjit Singh';
"""
cursor.execute(update_typo_sql)
conn.commit()
print(f"Rows updated: {cursor.rowcount}")

df_t3 = pd.read_sql_query("SELECT id, song_name, artist FROM Playlist WHERE artist = 'Arijit Singh';", conn)
display(df_t3)

### Task 4: Delete Short Songs (duration < 120s)
> **Safety Warning:** Always pair `DELETE` with a precise `WHERE` clause. Running `DELETE FROM Playlist;` without a `WHERE` condition wipes out all table rows.

In [ ]:
delete_short_sql = """
DELETE FROM Playlist
WHERE duration < 120;
"""
cursor.execute(delete_short_sql)
conn.commit()
print(f"Rows deleted: {cursor.rowcount}")

df_t4 = pd.read_sql_query("SELECT * FROM Playlist;", conn)
display(df_t4)

### Task 5: Append ' (Remix)' to AP Dhillon Songs (duration > 180s)
Demonstrates compound `WHERE` conditions (`artist = 'AP Dhillon' AND duration > 180`). Tracks with `duration <= 180` (such as 'With You' at 150s) remain untouched.

In [ ]:
remix_sql = """
UPDATE Playlist
SET song_name = song_name || ' (Remix)'
WHERE artist = 'AP Dhillon' AND duration > 180;
"""
cursor.execute(remix_sql)
conn.commit()
print(f"Remix tracks updated: {cursor.rowcount}")

df_t5 = pd.read_sql_query("SELECT id, song_name, artist, duration FROM Playlist WHERE artist = 'AP Dhillon';", conn)
display(df_t5)

### Final Playlist Inspection

In [ ]:
df_final = pd.read_sql_query("SELECT id, song_name, artist, duration, created_at FROM Playlist ORDER BY id;", conn)
display(df_final)
conn.close()